In [ ]:
import json
from pathlib import Path

from xdomain_ser.core import data_helper as dh, ser
from xdomain_ser.cli import (
    _load_extractor,
    _extract_one,
    _ser_report,
    _resolve_checkpoint_dir,
    _resolve_prompt_examples_path,
)

## Step 1: write the hint map

A hint map JSON has a single top-level key per domain (we prefix it with `hm_` by convention), and each domain entry has a `hint_map_id` repeated for downstream code and a `hint_map` dict mapping slot names to free-text descriptions. The descriptions land in the LoRA prompt as the domain schema, so write them like you'd brief a human annotator.

In [ ]:
thermostat_hint_map = {
    "hm_thermostat": {
        "hint_map_id": "hm_thermostat",
        "hint_map": {
            "target_temp": "target temperature in Fahrenheit (e.g., 68, 72)",
            "hvac_mode": "HVAC mode (heat, cool, auto, off)",
            "time_of_day": "schedule slot (morning, afternoon, evening, night)",
            "zone": "house zone (living room, bedroom, basement, whole house)",
        },
    }
}

# Save next to the notebook so the rest of the cells can use it.
out_path = Path("thermostat_hint_map.json")
with out_path.open("w") as f:
    json.dump(thermostat_hint_map, f, indent=2)
print(f"Wrote {out_path}")

## Step 2: load the extractor with the custom hint map

We pass our custom JSON via the `hint_map_path` argument. The `prompt_examples_path` still points at the default few-shot pool; the LoRA's few-shot examples come from the multi-domain training data, not from our new domain.

In [ ]:
checkpoint = _resolve_checkpoint_dir(None)
prompt_examples_path = _resolve_prompt_examples_path(None)

model, tokenizer, hint_map_table, pool = _load_extractor(
    checkpoint, out_path, prompt_examples_path
)
print("Available domains:", list(hint_map_table.keys()))

## Step 3: extract MRs on a few thermostat utterances

Three short examples, ranging from very direct to slightly conversational.

In [ ]:
examples = [
    {
        "text": "Set the bedroom to 68 in heat mode at night.",
        "gold": {"target_temp": "68", "hvac_mode": "heat", "time_of_day": "night", "zone": "bedroom"},
    },
    {
        "text": "In the morning, run the whole house on auto at 72 degrees.",
        "gold": {"target_temp": "72", "hvac_mode": "auto", "time_of_day": "morning", "zone": "whole house"},
    },
    {
        "text": "Keep the basement cool, around 70, in the afternoon.",
        "gold": {"target_temp": "70", "hvac_mode": "cool", "time_of_day": "afternoon", "zone": "basement"},
    },
]

for ex in examples:
    pred_str = _extract_one(ex["text"], "hm_thermostat", model, tokenizer, hint_map_table, pool)
    pred = dict(ser.extract_attributes_dict(pred_str.replace(dh.LIST_END, "")))
    # Flatten lists to scalars for SER comparison
    pred = {k: (v[0] if isinstance(v, list) and v else v) for k, v in pred.items()}
    report = _ser_report(pred, ex["gold"])
    print("-" * 60)
    print(f"Text:       {ex['text']}")
    print(f"Gold:       {ex['gold']}")
    print(f"Predicted:  {pred}")
    print(f"SER: {report['SER']:.3f}  Slot F1: {report['slot_f1']:.3f}")

## Step 4: iterate on the hint map

If the extraction misfires — say, the LoRA emits the temperature with a `°F` suffix or treats "basement" as both `zone` and a hallucinated slot — update the slot description to constrain the value space. The description is the only signal the LoRA gets about how to format values, so being explicit pays off.

Example refinement:

In [ ]:
refined_hint_map = {
    "hm_thermostat": {
        "hint_map_id": "hm_thermostat",
        "hint_map": {
            "target_temp": "target temperature as a bare integer in Fahrenheit, no units (e.g., '68', '72')",
            "hvac_mode": "one of: heat, cool, auto, off",
            "time_of_day": "one of: morning, afternoon, evening, night",
            "zone": "house zone, e.g., 'living room', 'bedroom', 'basement', 'whole house'",
        },
    }
}
with out_path.open("w") as f:
    json.dump(refined_hint_map, f, indent=2)
print("Refined hint map written; re-load the extractor and re-run the cell above to see if extractions improve.")

## Caveats

* This is genuine zero-shot transfer. Slot names that look like the LoRA's training schemas (descriptive attributes for venues, products, services) tend to transfer well. Slot names with enumerated value sets the LoRA has never seen (e.g., chemical species, instrument codes) may need a brief fine-tune to perform reliably.
* If you have ~50 labeled examples for the new domain, fine-tuning the extractor on them via `xdomain_ser.extraction.train` is fast (~30 min on RTX 3090) and substantially improves quality. The fine-tune does *not* require regenerating the existing multi-domain training set — just add your new examples and re-train.
* The hint map JSON can also include a `value_map` field (slot → list of valid enumerated values) if you want to constrain the LoRA's outputs at decode time. See `data/multi_ser_v9/hint_maps_v4.json` for examples (ViGGO uses this for ESRB ratings).